# vaastav-fpl reconnaissance

First look at the dataset before building anything on it. Source:
https://github.com/vaastav/Fantasy-Premier-League, pinned in
`references/data-sources/vaastav-fpl.md`.

In [1]:
from pathlib import Path

import polars as pl

VAASTAV_DIR = Path.cwd().parents[1] / "data" / "external" / "vaastav-fpl" / "data"

SEASONS = sorted(
    p.name for p in VAASTAV_DIR.iterdir() if p.is_dir() and (p / "gws" / "merged_gw.csv").exists()
)
SEASONS

['2016-17',
 '2017-18',
 '2018-19',
 '2019-20',
 '2020-21',
 '2021-22',
 '2022-23',
 '2023-24',
 '2024-25',
 '2025-26']

Ten seasons with gameweek data. (2026-27 exists in the repo but has no completed gameweeks
yet — season just started.) Let's start with the file the whole model depends on.

In [2]:
def read_merged_gw(season: str) -> pl.DataFrame:
    return pl.read_csv(
        VAASTAV_DIR / season / "gws" / "merged_gw.csv",
        infer_schema_length=0,
        ignore_errors=True,
        encoding="utf8-lossy",
    )


gw = read_merged_gw("2024-25")
gw.head()

name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,fixture,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,mng_clean_sheets,mng_draw,mng_goals_scored,mng_loss,mng_underdog_draw,mng_underdog_win,mng_win,modified,opponent_team,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,starts,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Alex Scott""","""MID""","""Bournemouth""","""1.6""","""0""","""0""","""11""","""0""","""12.8""","""77""","""0.01""","""0.01""","""0.00""","""1.02""","""6""","""1""","""0""","""3.6""","""22.8""","""2024-08-17T14:00:00Z""","""62""",null,null,null,null,null,null,null,"""False""","""16""","""0""","""0""","""0""","""0""","""1""","""0""","""4339""","""1""","""1""","""1""","""0.0""","""2""","""0""","""0""","""0""","""50""","""False""","""0""","""1"""
"""Carlos Miguel dos Santos Perei…","""GK""","""Nott'm Forest""","""2.2""","""0""","""0""","""0""","""0""","""0.0""","""427""","""0.00""","""0.00""","""0.00""","""0.00""","""6""","""0""","""0""","""0.0""","""0.0""","""2024-08-17T14:00:00Z""","""0""",null,null,null,null,null,null,null,"""False""","""3""","""0""","""0""","""0""","""0""","""1""","""0""","""33324""","""0""","""1""","""1""","""0.0""","""0""","""0""","""0""","""0""","""45""","""True""","""0""","""1"""
"""Tomiyasu Takehiro""","""DEF""","""Arsenal""","""0.0""","""0""","""0""","""0""","""0""","""0.0""","""22""","""0.00""","""0.00""","""0.00""","""0.00""","""2""","""0""","""0""","""0.0""","""0.0""","""2024-08-17T14:00:00Z""","""0""",null,null,null,null,null,null,null,"""False""","""20""","""0""","""0""","""0""","""0""","""1""","""0""","""8462""","""0""","""0""","""2""","""0.0""","""0""","""0""","""0""","""0""","""50""","""True""","""0""","""1"""
"""Malcolm Ebiowei""","""MID""","""Crystal Palace""","""0.0""","""0""","""0""","""0""","""0""","""0.0""","""197""","""0.00""","""0.00""","""0.00""","""0.00""","""8""","""0""","""0""","""0.0""","""0.0""","""2024-08-18T13:00:00Z""","""0""",null,null,null,null,null,null,null,"""False""","""4""","""0""","""0""","""0""","""0""","""1""","""0""","""716""","""0""","""1""","""2""","""0.0""","""0""","""0""","""0""","""0""","""45""","""False""","""0""","""1"""
"""Ben Brereton Díaz""","""MID""","""Southampton""","""1.0""","""0""","""0""","""-2""","""0""","""14.0""","""584""","""0.02""","""0.32""","""0.30""","""0.25""","""5""","""1""","""0""","""3.3""","""2.6""","""2024-08-17T14:00:00Z""","""70""",null,null,null,null,null,null,null,"""False""","""15""","""0""","""0""","""0""","""0""","""1""","""0""","""66244""","""1""","""0""","""1""","""16.0""","""1""","""0""","""0""","""0""","""55""","""False""","""1""","""1"""


First question: does this really give one row per player per gameweek, like `merged_gw`
implies? If it does, `(element, GW)` should be a unique key.

In [3]:
dupes = gw.group_by(["element", "GW"]).len().filter(pl.col("len") > 1)
dupes.height, dupes.sort("len", descending=True).head()

(374,
 shape: (5, 3)
 ┌─────────┬─────┬─────┐
 │ element ┆ GW  ┆ len │
 │ ---     ┆ --- ┆ --- │
 │ str     ┆ str ┆ u32 │
 ╞═════════╪═════╪═════╡
 │ 31      ┆ 25  ┆ 2   │
 │ 704     ┆ 33  ┆ 2   │
 │ 666     ┆ 33  ┆ 2   │
 │ 206     ┆ 33  ┆ 2   │
 │ 394     ┆ 32  ┆ 2   │
 └─────────┴─────┴─────┘)

374 duplicate keys in 2024-25 alone — not a handful of data-entry errors. Before assuming
messy data, check what's actually different between the rows sharing a key.

In [4]:
worst = dupes.sort("len", descending=True).row(0, named=True)
sample_element, sample_gw = worst["element"], worst["GW"]

gw.filter((pl.col("element") == sample_element) & (pl.col("GW") == sample_gw)).select(
    "name", "GW", "fixture", "opponent_team", "kickoff_time", "minutes", "total_points"
)

name,GW,fixture,opponent_team,kickoff_time,minutes,total_points
str,str,str,str,str,str,str
"""Emiliano Buendía Stati""","""25""","""241""","""10""","""2025-02-15T15:00:00Z""","""0""","""0"""
"""Emiliano Buendía Stati""","""25""","""282""","""12""","""2025-02-19T19:30:00Z""","""0""","""0"""


Two different `fixture` ids, two different opponents, same `GW` number — a double gameweek.
So `(element, GW)` isn't the grain; `(element, GW, fixture)` is, i.e. player-fixture. Check
that holds for every season, not just 2024-25.

In [5]:
grain_summary = []
for season in SEASONS:
    df = read_merged_gw(season)
    d = df.group_by(["element", "GW"]).len().filter(pl.col("len") > 1)
    grain_summary.append(
        {
            "season": season,
            "rows": df.height,
            "dgw_keys": d.height,
            "max_rows_per_key": d["len"].max() if d.height else 1,
        }
    )

pl.DataFrame(grain_summary)

season,rows,dgw_keys,max_rows_per_key
str,i64,i64,i64
"""2016-17""",23679,573,2
"""2017-18""",22467,670,2
"""2018-19""",21790,656,2
"""2019-20""",22560,247,2
"""2020-21""",24365,1437,3
"""2021-22""",25447,2217,2
"""2022-23""",26505,1548,2
"""2023-24""",29725,983,2
"""2024-25""",27605,374,2


Consistent across every season — DGWs are a normal, recurring feature, not a one-off in one
season's data. Grain confirmed: player-fixture.

Now the flip side — blank gameweeks. If a team doesn't play, does a player still get a row
with `minutes=0`, or does the row disappear entirely? How this behaves determines whether a
blank gameweek can be scored as an observed zero or has to be excluded from error metrics.

In [6]:
fixtures = pl.read_csv(
    VAASTAV_DIR / "2024-25" / "fixtures.csv", infer_schema_length=0, encoding="utf8-lossy"
)
long = pl.concat(
    [
        fixtures.select(pl.col("team_h").alias("team"), "event"),
        fixtures.select(pl.col("team_a").alias("team"), "event"),
    ]
).drop_nulls("event")

teams = sorted(long["team"].unique().to_list())
season_gws = sorted({int(g) for g in long["event"].unique().to_list()})
grid = pl.DataFrame(
    {"team": [t for t in teams for _ in season_gws], "event": [g for _ in teams for g in season_gws]}
).with_columns(pl.col("event").cast(pl.Utf8))
fixture_counts = grid.join(
    long.group_by(["team", "event"]).len(), on=["team", "event"], how="left"
).with_columns(pl.col("len").fill_null(0))

blanks = fixture_counts.filter(pl.col("len") == 0)
blanks.head()

team,event,len
str,str,u32
"""1""","""34""",0
"""12""","""15""",0
"""12""","""29""",0
"""13""","""34""",0
"""15""","""29""",0


Pick one of these and check: does that team's squad show up in the gameweek data at all?

In [7]:
blank_team, blank_gw = blanks.row(0, named=True)["team"], blanks.row(0, named=True)["event"]

raw = pl.read_csv(
    VAASTAV_DIR / "2024-25" / "players_raw.csv",
    infer_schema_length=0,
    ignore_errors=True,
    encoding="utf8-lossy",
)
team_players = set(raw.filter(pl.col("team") == blank_team)["id"].to_list())
players_with_a_row = set(gw.filter(pl.col("GW") == blank_gw)["element"].to_list())
missing = team_players - players_with_a_row

f"team {blank_team}, GW {blank_gw}: {len(missing)} of {len(team_players)} squad players have no row at all"

'team 1, GW 34: 37 of 37 squad players have no row at all'

All of them missing, not zero-value rows. A blank gameweek is an absent row for every player
on that team. So a blank gameweek can carry xP=0 for squad selection while still being
excluded from error metrics, rather than scored as a predicted zero.

Next: is keying walk-forward cutoffs on `kickoff_time` instead of `GW` number actually
necessary in this data, or just defensive caution?

In [8]:
ordering_summary = []
for season in SEASONS:
    df = read_merged_gw(season)
    nulls = df["kickoff_time"].null_count()
    parsed = df.with_columns(
        pl.col("kickoff_time")
        .str.to_datetime(format="%Y-%m-%dT%H:%M:%SZ", time_zone="UTC", strict=False)
        .alias("ko"),
        pl.col("GW").cast(pl.Int64, strict=False).alias("gw_int"),
    ).drop_nulls(["ko", "gw_int"])
    bounds = (
        parsed.group_by("gw_int")
        .agg(pl.col("ko").min().alias("lo"), pl.col("ko").max().alias("hi"))
        .sort("gw_int")
    )
    violations = 0
    running_max = None
    for row in bounds.iter_rows(named=True):
        if running_max is not None and row["lo"] < running_max:
            violations += 1
        running_max = row["hi"] if running_max is None else max(running_max, row["hi"])
    ordering_summary.append(
        {"season": season, "kickoff_nulls": nulls, "gw_ordering_violations": violations}
    )

pl.DataFrame(ordering_summary)

season,kickoff_nulls,gw_ordering_violations
str,i64,i64
"""2016-17""",0,0
"""2017-18""",0,0
"""2018-19""",0,0
"""2019-20""",0,0
"""2020-21""",0,0
"""2021-22""",0,0
"""2022-23""",0,0
"""2023-24""",0,0
"""2024-25""",0,0


Zero nulls, zero ordering violations, in every season. In this particular dataset `GW` would
actually sort correctly on its own. Doesn't change the plan though: `kickoff_time` is still
the right key, because it's the only thing that disambiguates two fixtures sharing the same
`GW` number in a DGW, which `GW` alone can't do.

Now the column-availability question, since half the model's components depend on
xG/xA/starts/defcon being there at all. Building a season × column table rather than trusting
the data dictionary's word for it.

In [9]:
watch_cols = [
    "expected_goals",
    "expected_assists",
    "expected_goals_conceded",
    "starts",
    "clearances_blocks_interceptions",
    "tackles",
    "recoveries",
    "defensive_contribution",
    "position",
    "xP",
]

presence = []
for season in SEASONS:
    df = read_merged_gw(season)
    row = {"season": season}
    for c in watch_cols:
        row[c] = "-" if c not in df.columns else f"{df[c].null_count() / df.height:.0%} null"
    presence.append(row)

pl.DataFrame(presence)

season,expected_goals,expected_assists,expected_goals_conceded,starts,clearances_blocks_interceptions,tackles,recoveries,defensive_contribution,position,xP
str,str,str,str,str,str,str,str,str,str,str
"""2016-17""","""-""","""-""","""-""","""-""","""0% null""","""0% null""","""0% null""","""-""","""-""","""-"""
"""2017-18""","""-""","""-""","""-""","""-""","""0% null""","""0% null""","""0% null""","""-""","""-""","""-"""
"""2018-19""","""-""","""-""","""-""","""-""","""0% null""","""0% null""","""0% null""","""-""","""-""","""-"""
"""2019-20""","""-""","""-""","""-""","""-""","""-""","""-""","""-""","""-""","""-""","""-"""
"""2020-21""","""-""","""-""","""-""","""-""","""-""","""-""","""-""","""-""","""0% null""","""0% null"""
"""2021-22""","""-""","""-""","""-""","""-""","""-""","""-""","""-""","""-""","""0% null""","""0% null"""
"""2022-23""","""0% null""","""0% null""","""0% null""","""0% null""","""-""","""-""","""-""","""-""","""0% null""","""0% null"""
"""2023-24""","""0% null""","""0% null""","""0% null""","""0% null""","""-""","""-""","""-""","""-""","""0% null""","""0% null"""
"""2024-25""","""0% null""","""0% null""","""0% null""","""0% null""","""-""","""-""","""-""","""-""","""0% null""","""0% null"""


xG, xA, xGC, and `starts` all begin in 2022-23. Raw defensive counts
(`clearances_blocks_interceptions`, `tackles`, `recoveries`) are present 2016-17 to 2018-19,
absent for six seasons (2019-20 to 2024-25), then return in 2025-26 alongside the new
`defensive_contribution` scoring column — a gap, not one continuous pre-history. `position`
and `xP` both start in 2020-21.

Is `xP` safe to use, even as a benchmark?

In [10]:
dict_text = (VAASTAV_DIR.parent / "DATA_DICTIONARY.md").read_text(encoding="utf-8")
print(dict_text.split("Caveat on `xP`")[1][:650])

 (timing uncertainty):** `xP` is scraped from FPL's `ep_this`
> field *after* each gameweek has ended. FPL's update cadence for this field is
> not documented, and empirical evidence suggests scraped values may reflect
> post-match information rather than the pre-match prediction managers actually
> saw before the deadline. If you are using `xP` as an ML feature, treat it as
> potentially post-match (apply `shift(1)` within each `element` group, or drop
> the column). See the "Known Data Limitations" / "xP column" section of the
> README before relying on it.

### Defensive Stats (when available)

| Column | Type | Description |
|--------|---


The source repo's own documentation raises the same concern: `xP` is scraped from FPL's
`ep_this` field *after* the gameweek ends, and the scrape timing isn't guaranteed to be
pre-deadline. Dropping the column outright rather than lagging it — a `shift` across the
DGW/blank grain is an easy place for a silent bug.

Now the trickier one: cross-season player identity. `element` is supposed to be season-scoped.
Checking what happens when that gets ignored, joining directly on `element` across all ten
seasons and using position (`element_type`) as a consistency check.

In [11]:
per_season_element = {}
for season in SEASONS:
    raw = pl.read_csv(
        VAASTAV_DIR / season / "players_raw.csv",
        infer_schema_length=0,
        ignore_errors=True,
        encoding="utf8-lossy",
    )
    per_season_element[season] = raw.select(pl.col("id").alias("element"), pl.col("element_type").alias(season))

joined = per_season_element[SEASONS[0]]
for s in SEASONS[1:]:
    joined = joined.join(per_season_element[s], on="element", how="inner")

reclassified = sum(1 for r in joined.iter_rows(named=True) if len({r[s] for s in SEASONS}) > 1)
joined.height, reclassified, f"{reclassified / joined.height:.0%}"

(620, 619, '100%')

...100% reclassified? Arsenal did not field several hundred outfield players who all switched
position over a decade. `element` gets reused across seasons for different real players, and a
naive join on it silently treats them as one person.

`players_raw.csv` also carries a `code` field, the actual stable FPL player identifier.
Checking whether that behaves better.

In [12]:
per_season_code = {}
for season in SEASONS:
    raw = pl.read_csv(
        VAASTAV_DIR / season / "players_raw.csv",
        infer_schema_length=0,
        ignore_errors=True,
        encoding="utf8-lossy",
    )
    per_season_code[season] = raw.select("code", pl.col("element_type").alias(season))

joined_code = per_season_code[SEASONS[0]]
for s in SEASONS[1:]:
    joined_code = joined_code.join(per_season_code[s], on="code", how="inner")

reclassified_code = sum(1 for r in joined_code.iter_rows(named=True) if len({r[s] for s in SEASONS}) > 1)
joined_code.height, reclassified_code, f"{reclassified_code / joined_code.height:.0%}"

(35, 5, '14%')

35 players survived all ten seasons under the `code` key. 5 of them (14%) show a differing
`element_type` in at least one season. `code`, not `element`, is the cross-season key, and it
lives in `players_raw.csv` — not in `player_idlist.csv`, which (checked separately) only has
`first_name, second_name, id`, no `code` at all, despite the name suggesting it's the
id-mapping file.

The player crosswalk needs to be built from `players_raw.csv.code` per season, not from the
file whose name implies it should be there.

Last check: is set-piece and penalty duty absent from this dataset, as assumed going in?

In [13]:
watch_setpiece = ["penalties_order", "direct_freekicks_order", "corners_and_indirect_freekicks_order"]
[c for c in watch_setpiece if c in gw.columns]

[]

Nothing in the player-fixture file. But that's not the only file in the source repo — checking
`players_raw.csv`, the season-level snapshot, before concluding it's absent.

In [14]:
raw_2526 = pl.read_csv(
    VAASTAV_DIR / "2025-26" / "players_raw.csv",
    infer_schema_length=0,
    ignore_errors=True,
    encoding="utf8-lossy",
)
[c for c in watch_setpiece if c in raw_2526.columns]

['penalties_order',
 'direct_freekicks_order',
 'corners_and_indirect_freekicks_order']

It's there. Checking whether it's actually populated, or an empty column carried over from
the API schema.

In [15]:
is_missing = pl.col("penalties_order").is_null() | (pl.col("penalties_order") == "None")
raw_2526.filter(~is_missing).select("web_name", *watch_setpiece).head(10)

web_name,penalties_order,direct_freekicks_order,corners_and_indirect_freekicks_order
str,str,str,str
"""Saka""","""1""","""2""","""2"""
"""Ødegaard""","""3""","""None""","""4"""
"""Trossard""","""4""","""None""","""None"""
"""Gyökeres""","""2""","""None""","""None"""
"""Buendía""","""2""","""6""","""None"""
"""Watkins""","""3""","""None""","""None"""
"""Bruun Larsen""","""3""","""2""","""None"""
"""Flemming""","""1""","""5""","""None"""
"""Barnes""","""2""","""None""","""None"""


Real values — Saka as primary penalty taker, Ødegaard on corners. One catch:
`players_raw.csv` lives at `data/<season>/`, not inside `gws/`. It's a single current snapshot
of "today's" state, not a per-gameweek history. There's one file per season, not one per
gameweek, so there's no way to recover who had penalty duty in, say, GW10 specifically, if it
changed since. Useful as a live feature when predicting forward off a fresh scrape, not usable
for walk-forward backtesting without an archived-per-gameweek version, which this repo doesn't
have.

In [16]:
first_season_setpiece = next(
    season
    for season in SEASONS
    if "penalties_order"
    in pl.read_csv(
        VAASTAV_DIR / season / "players_raw.csv",
        infer_schema_length=0,
        ignore_errors=True,
        encoding="utf8-lossy",
    ).columns
)
first_season_setpiece

'2020-21'

Set-piece and penalty duty is not fully absent from this dataset: `players_raw.csv` has it
from 2020-21 onward, current-snapshot-only, usable live but not for backtesting.

Full write-up, reproduction command, and file hash manifest:
`references/data-sources/vaastav-fpl.md`.